In [ ]:
# 1/4 - clone + dotnet (fault-tolerant)
# 1/4 - clone + dotnet (fault-tolerant)
print('===== ALL NATIVE KERNELS -> PURE TKV (14+7+2) + LOSS CHECK =====')
# 1/4 - clone main + prototype + Inference, install .NET (fault-tolerant)
import subprocess, os, shutil
print('===== INTEGRATE 14 PURE TKV KERNELS INTO NATIVE BUILD + LOSS CHECK =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
subprocess.run('rm -rf /content/TVinf', shell=True)
subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git /content/TVinf', shell=True, capture_output=True, text=True, timeout=600)
dotnet=shutil.which('dotnet')
if not dotnet:
    print('dotnet not found -> installing .NET 10 SDK...')
    p=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && chmod +x /tmp/dotnet-install.sh && /tmp/dotnet-install.sh --channel 10.0 --install-dir /usr/share/dotnet', shell=True, capture_output=True, text=True, timeout=900)
    print('install rc=',p.returncode)
    dotnet='/usr/share/dotnet/dotnet'
print('dotnet =',dotnet,'exists=',os.path.exists(dotnet))

In [ ]:
# 1/4 - clone main + prototype (fresh)
# 2/4 - build TOAN BO native kernel tu .tkv THUAN (43 exports) + copy .so
import subprocess, shutil, os, re
root='/content/TVmain/Release/tools/simd_native'
ex='/content/TVcpu/Release/cuda_backend/examples'
bwd=['bias_add_rows_cpu.cpu.c','softmax_causal_fwd_pure.cpu.c','merge_hb_head_cpu.cpu.c',
     'layernorm_bwd_cpu.cpu.c','gelu_erf_bwd_pure.cpu.c','gelu_erf_fwd_pure.cpu.c',
     'bias_grad_cpu.cpu.c','softmax_causal_bwd_pure.cpu.c','embed_bwd_pure.cpu.c',
     'layernorm_fwd_cpu.cpu.c']
parts=[]
for f in bwd:
    s=open(ex+'/'+f).read()
    ls=[l for l in s.split('\n') if not l.startswith('#include') and not l.startswith('#define') and l.strip()!='']
    parts.append('\n'.join(ls))
hdr='#include <immintrin.h>\n#include <math.h>\n#include <stdlib.h>\n#define EXPORT __attribute__((visibility("default")))\n'
open(root+'/tkv_bwd_ops.c','w').write(hdr+'\n\n'.join(parts)+'\n')
shutil.copyfile(ex+'/linear_dw_cpu.cpu.c', root+'/tkv_linear_dw.c')
shutil.copyfile(ex+'/linear_dx_cpu.cpu.c', root+'/tkv_linear_dx.c')
p=root+'/tkv_simd_native.c'; s=open(p).read()
for name,gf in [('F32ListAdd','f32_list_add_cpu.cpu.c'),('F32ListCopy','f32_list_copy_cpu.cpu.c')]:
    s=re.sub(r'EXPORT int '+name+r'\(.*?\n\}\n', open(ex+'/'+gf).read().strip()+'\n', s, count=1, flags=re.S)
open(p,'w').write(s)
shutil.copyfile(ex+'/flash_all_cpu.cpu.c', root+'/tkv_flash_attn.c')
shutil.copyfile(ex+'/attn_cuda_stub_cpu.cpu.c', root+'/tkv_attn_cuda.c')
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
r=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('ALL-TKV native build rc=',r.returncode)
assert r.returncode==0, r.stderr[:3000]
shutil.copyfile(root+'/tkv_simd_native.so','/content/TVinf/demo/tkv_simd_native.so')
print('.so copied to Inference demo')

In [ ]:
# 2/4 - replicate gpt2_init(7) cua NATIVE bang numpy f32, nap vao HF, so loss
import numpy as np, torch, math
from transformers import GPT2Config, GPT2LMHeadModel
T=1024; V=50257; NL=12; H=768; NH=12; HD=64; SEED=7
f=np.float32
def gi_noise(n, seed):
    # Distribution-equivalent (N(0, 0.02^2)) voi native _gi_noise, vectorized.
    # Khong bit-identical RNG, nhung CUNG SCALE -> du de biet init hay implementation
    # gay ra chenh loss 25.45 vs 10.8.
    g=np.random.default_rng(seed)
    return g.standard_normal(n).astype(np.float32)*np.float32(0.02)
def ones(n): return np.ones(n,dtype=np.float32)
def zeros(n): return np.zeros(n,dtype=np.float32)
sd={}
sd['transformer.wte.weight']=gi_noise(V*H,SEED+1).reshape(V,H)
sd['transformer.wpe.weight']=gi_noise(1024*H,SEED+2).reshape(1024,H)
sd['transformer.ln_f.weight']=ones(H); sd['transformer.ln_f.bias']=zeros(H)
for li in range(NL):
    s=SEED+10+li*100; p='transformer.h.%d.'%li
    sd[p+'ln_1.weight']=ones(H); sd[p+'ln_1.bias']=zeros(H)
    sd[p+'attn.c_attn.weight']=gi_noise(H*3*H,s+1).reshape(H,3*H)
    sd[p+'attn.c_attn.bias']=zeros(3*H)
    sd[p+'attn.c_proj.weight']=gi_noise(H*H,s+2).reshape(H,H)
    sd[p+'attn.c_proj.bias']=zeros(H)
    sd[p+'ln_2.weight']=ones(H); sd[p+'ln_2.bias']=zeros(H)
    sd[p+'mlp.c_fc.weight']=gi_noise(H*4*H,s+3).reshape(H,4*H)
    sd[p+'mlp.c_fc.bias']=zeros(4*H)
    sd[p+'mlp.c_proj.weight']=gi_noise(4*H*H,s+4).reshape(4*H,H)
    sd[p+'mlp.c_proj.bias']=zeros(H)
cfg=GPT2Config(n_layer=NL,n_head=NH,n_embd=H,n_positions=1024,vocab_size=V,
               layer_norm_epsilon=1e-6,activation_function='gelu',
               resid_pdrop=0.0,attn_pdrop=0.0,embd_pdrop=0.0)
model=GPT2LMHeadModel(cfg).eval()
model.load_state_dict({k:torch.from_numpy(v.copy()) for k,v in sd.items()})
ids=np.array([(i*37+11)%V for i in range(T)],dtype=np.int64)
xi=torch.from_numpy(ids).unsqueeze(0)
with torch.no_grad():
    lg=model(input_ids=xi).logits[0].numpy()
# native-style CE: mean_i(lse_i - logit[i, ids[i]])
mx=lg.max(axis=1)
ce=(mx+np.log(np.exp(lg-mx[:,None]).sum(axis=1))-lg[np.arange(T),ids]).mean()
print('HF loss with NATIVE-init weights = %.6f'%ce)
print('NATIVE bseq loss (same init)     = 25.453860')
print('DIFF = %.6f'%abs(ce-25.453860))
print('ln(V) = %.6f'%np.log(V))

In [ ]:
# 3/4 - summary
print('If HF with native-init weights ~ 10.8 but native says 25.45, difference is in forward/loss implementation, not init.')